
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [27]:
import os 
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")

In [28]:
from pydantic import BaseModel, Field 
class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")
    review:str=Field(description="A review of the movie")
    plot:str=Field(description="The plot of the movie")
    cast:list[str]=Field(description="The cast of the movie")
    genre:list[str]=Field(description="The genre of the movie")


In [29]:
model_with_structured_output = model.with_structured_output(Movie)

In [30]:
model_with_structured_output

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002046D3FC800>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002046E80A720>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The titl

In [31]:
model.invoke("Proved details about the movie Inception")

AIMessage(content='Below is a comprehensive “fact‑sheet” on Christopher\u202fNolan’s 2010 science‑fiction thriller **Inception**.  \nEverything is drawn from publicly available sources (film credits, press releases, box‑office data, interviews, reviews, etc.) and is presented in a neutral, factual style.\n\n---\n\n## 1. Basic Production Information\n\n| Item | Detail |\n|------|--------|\n| **Title** | Inception |\n| **Director** | Christopher Nolan |\n| **Producer** | Emma Thomas (Nolan’s wife), Christopher Nolan |\n| **Screenplay** | Christopher Nolan |\n| **Story** | Christopher Nolan |\n| **Music** | Hans\u202fZimmer (original score) |\n| **Cinematography** | Wally\u202fPfister |\n| **Editing** | Lee\u202fSmith |\n| **Production Companies** | Syncopy, Warner Bros. Pictures, Legendary Pictures |\n| **Distributor** | Warner Bros. Pictures |\n| **Runtime** | 2\u202fh\u202f28\u202fmin (148\u202fmin) |\n| **Country** | United States |\n| **Language** | English |\n| **Budget** | $160\u20

In [32]:
model_with_structured_output.invoke("Proved details about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8, review="Inception is a mind-bending thriller that combines a complex narrative structure with stunning visual effects, showcasing Nolan's mastery of storytelling and his ability to blend action with philosophical questions about reality and perception.", plot='A thief who steals corporate secrets through the use of dream-sharing technology is offered a chance to have his criminal history erased if he can implant an idea into the mind of a target.', cast=['Leonardo DiCaprio', 'Joseph Gordon-Levitt', 'Elliot Page', 'Tom Hardy', 'Ken Watanabe', 'Cillian Murphy', 'Tom Berenger', 'Marion Cotillard'], genre=['Action', 'Adventure', 'Sci-Fi'])

In [33]:
response = model_with_structured_output.invoke("Proved details about the movie Obession 2026")
response

Movie(title='Obession', year=2026, director='Kathryn Bigelow', rating=8.2, review="Critics praised the film's intense pacing, atmospheric tension, and Emma Stone's nuanced performance as she navigates the thin line between love and fixation. The direction by Kathryn Bigelow keeps viewers on edge, while the screenplay offers a fresh take on psychological thrillers.", plot='A gripping tale of a woman whose obsessive love for her husband leads to a series of escalating events, blurring the line between devotion and control. The film explores the psychological toll of obsession and the desperate measures taken to reclaim autonomy.', cast=['Emma Stone', 'Daniel Kaluuya', "Lupita Nyong'o"], genre=['Thriller', 'Psychological', 'Drama'])

In [34]:
response.rating

8.2

In [35]:
response.cast

['Emma Stone', 'Daniel Kaluuya', "Lupita Nyong'o"]

### Message output alongside parsed structure

In [36]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to provide details about the movie Inception. The user likely wants director, rating, title, year. We can call the function. Use functions.Movie with director: Christopher Nolan, rating: maybe 8.8? The rating out of 10. Title: Inception. Year: 2010.', 'tool_calls': [{'id': 'fc_3e221249-3821-48e1-823e-177c9e9d531d', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 164, 'total_tokens': 268, 'completion_time': 0.106492717, 'completion_tokens_details': {'reasoning_tokens': 65}, 'prompt_time': 0.009251895, 'prompt_tokens_details': None, 'queue_time': 0.320293401, 'total_time': 0.115744612}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9b8528b477', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None

### Nested Structure

In [37]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

In [38]:
model_structure= model.with_structured_output(MovieDetails)

In [39]:
response = model_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal')], genres=['Action', 'Adventure', 'Sci-Fi'], budget=160000000.0)

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [40]:
from typing_extensions import TypedDict,Annotated

class MoveDict(TypedDict):
    """A movie with details """
    title: Annotated[str, "The title of the movie"]
    year: Annotated[int, "The year the movie was released"]
    director: Annotated[str, "The director of the movie"]
    rating: Annotated[float, "The movie's rating out of 10"]

In [41]:
model_struc_dict = model.with_structured_output(MoveDict)

In [42]:
response = model_struc_dict.invoke("Provide details about the movie Avengers")

In [43]:
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [44]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_typedDict = model.with_structured_output(MovieDetails)
response = model_typedDict.invoke("Provide details about the movie Avengers")
response
   

{'budget': 356000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man (Tony Stark)'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'The Avengers',
 'year': 2008}

In [45]:
response = model_typedDict.invoke("Provide details about the Dexter amazon web series")
response

{'budget': 5000000,
 'cast': [{'name': 'Michael C. Hall', 'role': 'Dexter Morgan'},
  {'name': 'Jennifer Carpenter', 'role': 'Debra Morgan'},
  {'name': 'Julie Benz', 'role': 'Rita Bennett'},
  {'name': 'Catherine Keener', 'role': 'Lila Tournay'},
  {'name': 'David Zayas', 'role': 'Angel Batista'}],
 'genres': ['Crime', 'Drama', 'Thriller', 'Mystery'],
 'title': 'Dexter',
 'year': 2006}

In [46]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [47]:
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [48]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

In [49]:
agent = create_agent(
    model="gpt-3.5-turbo",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

In [50]:
result = agent.invoke({
    "messages":[
        {
            "role": "user",
            "content":  "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
        ]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='dc212222-f7e1-4c0c-a2c4-05b6835edd53'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 96, 'total_tokens': 128, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EVTPHyS1Gi9N4u7bmCOyhi7ASwaat', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a10a0b-df6e-7440-8600-c797d1ad9748-0', tool_calls=[{'name': 'ContactInfo', 'args': {'name': 'John Doe', 'email':

In [51]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [52]:
## TypedDict
from typing_extensions import TypedDict,Annotated

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-3.5-turbo",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages":[
        {
            "role": "user",
            "content":  "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
        ]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='3fc45ae4-f6fc-4260-80ab-ac24028fb087'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 72, 'total_tokens': 104, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EVTPMULVLISIkzlml7qSIagphUkux', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a10a0b-f920-7562-a2c7-8f15421bfed2-0', tool_calls=[{'name': 'ContactInfo', 'args': {'name': 'John Doe', 'email':

In [53]:
result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [54]:
## DataClass

from dataclasses import dataclass
from langchain.agents import create_agent


@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-3.5-turbo",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages":[
        {
            "role": "user",
            "content":  "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
        ]
})

result["structured_response"]


ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')